<a href="https://colab.research.google.com/github/jintubhuyan-2000/Spatial-Gradient-of-Highway-Induced-Land-Cover-Change/blob/main/SECTION_4_6_%E2%80%94_MAJOR_LULC_TRANSITION_PATHWAYS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =============================================================================
# TEZPUR–NORTH LAKHIMPUR NATIONAL HIGHWAY CORRIDOR
# SECTION 4.6 — MAJOR LULC TRANSITION PATHWAYS
#
# COMPLETE GEOTIFF-BASED TRANSITION ANALYSIS
#
# PERIOD:
#     2016 → 2025
#
# ORIGINAL LULC CLASSES
#     0 = Water
#     1 = Trees
#     2 = Grass
#     3 = Flooded Vegetation
#     4 = Crops
#     5 = Shrub/Scrub
#     6 = Built
#     7 = Bare
#     8 = Snow/Ice
#
# ANALYSIS RECLASSIFICATION
#     0       → Water
#     1 + 3   → Vegetation
#     2       → Grass
#     4       → Crops
#     5       → Shrub/Scrub
#     6       → Built
#     7 + 8   → Bare/Snow-Ice
#
# =============================================================================


import os
import math
import warnings

import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window

warnings.filterwarnings("ignore")


# =============================================================================
# 1. USER SETTINGS
# =============================================================================

INPUT_DIR = "/content/drive/MyDrive/TZPR_NLP_Research"

OUTPUT_DIR = os.path.join(
    INPUT_DIR,
    "LULC_Transition_Analysis_4_6"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

CHUNK_ROWS = 256


# =============================================================================
# 2. INPUT FILES
# =============================================================================

LULC_2016 = os.path.join(
    INPUT_DIR,
    "TZPR_NLP_LULC_2016.tif"
)

LULC_2025 = os.path.join(
    INPUT_DIR,
    "TZPR_NLP_LULC_2025.tif"
)


# =============================================================================
# 3. OUTPUT FILES
# =============================================================================

TRANSITION_RASTER = os.path.join(
    OUTPUT_DIR,
    "TZPR_NLP_LULC_Transition_2016_2025.tif"
)

TRANSITION_CSV = os.path.join(
    OUTPUT_DIR,
    "TZPR_NLP_LULC_Transition_Statistics.csv"
)

TRANSITION_MATRIX_CSV = os.path.join(
    OUTPUT_DIR,
    "TZPR_NLP_LULC_Transition_Matrix_ha.csv"
)

TRANSITION_PERCENT_CSV = os.path.join(
    OUTPUT_DIR,
    "TZPR_NLP_LULC_Transition_Matrix_Percent.csv"
)

RECLASS_2016_RASTER = os.path.join(
    OUTPUT_DIR,
    "TZPR_NLP_LULC_Reclassified_2016.tif"
)

RECLASS_2025_RASTER = os.path.join(
    OUTPUT_DIR,
    "TZPR_NLP_LULC_Reclassified_2025.tif"
)

SUMMARY_TXT = os.path.join(
    OUTPUT_DIR,
    "TZPR_NLP_LULC_Transition_Manuscript_Values.txt"
)


# =============================================================================
# 4. ANALYSIS CLASS DEFINITIONS
# =============================================================================

CLASS_NAMES = {

    0: "Water",

    1: "Vegetation",

    2: "Grass",

    3: "Crops",

    4: "Shrub/Scrub",

    5: "Built",

    6: "Bare/Snow-Ice",
}


# Original → analysis class

RECLASS_MAP = {

    0: 0,       # Water

    1: 1,       # Trees → Vegetation

    2: 2,       # Grass

    3: 1,       # Flooded Vegetation → Vegetation

    4: 3,       # Crops

    5: 4,       # Shrub/Scrub

    6: 5,       # Built

    7: 6,       # Bare → Bare/Snow-Ice

    8: 6,       # Snow/Ice → Bare/Snow-Ice
}


# =============================================================================
# 5. EARTH RADIUS
# =============================================================================

EARTH_RADIUS = 6378137.0


# =============================================================================
# 6. PIXEL AREA
# =============================================================================

def calculate_row_pixel_area_km2(
    src,
    row
):

    transform = src.transform

    pixel_width_deg = abs(
        transform.a
    )

    pixel_height_deg = abs(
        transform.e
    )

    lon_width_rad = math.radians(
        pixel_width_deg
    )

    top_lat = (
        transform.f
        +
        row * transform.e
    )

    bottom_lat = (
        transform.f
        +
        (row + 1) * transform.e
    )

    lat1 = math.radians(
        min(
            top_lat,
            bottom_lat
        )
    )

    lat2 = math.radians(
        max(
            top_lat,
            bottom_lat
        )
    )

    area_m2 = (
        EARTH_RADIUS ** 2
        *
        lon_width_rad
        *
        (
            math.sin(lat2)
            -
            math.sin(lat1)
        )
    )

    return (
        area_m2 /
        1_000_000.0
    )


# =============================================================================
# 7. RECLASSIFICATION FUNCTION
# =============================================================================

def reclassify_array(
    arr
):

    output = np.full(
        arr.shape,
        255,
        dtype=np.uint8
    )

    for original, new in RECLASS_MAP.items():

        output[
            arr == original
        ] = new

    return output


# =============================================================================
# 8. CHECK INPUT FILES
# =============================================================================

print("=" * 80)
print("SECTION 4.6 — MAJOR LULC TRANSITION PATHWAYS")
print("=" * 80)

print()
print("Input directory:")
print(INPUT_DIR)

print()
print("Output directory:")
print(OUTPUT_DIR)

print()
print("Checking input files...")

if not os.path.exists(LULC_2016):

    raise FileNotFoundError(
        f"2016 LULC raster not found:\n{LULC_2016}"
    )

if not os.path.exists(LULC_2025):

    raise FileNotFoundError(
        f"2025 LULC raster not found:\n{LULC_2025}"
    )

print(
    "[OK] 2016:",
    os.path.basename(LULC_2016)
)

print(
    "[OK] 2025:",
    os.path.basename(LULC_2025)
)


# =============================================================================
# 9. OPEN AND CHECK RASTERS
# =============================================================================

with rasterio.open(LULC_2016) as src16, \
     rasterio.open(LULC_2025) as src25:

    print()
    print("=" * 80)
    print("RASTER VALIDATION")
    print("=" * 80)

    if src16.width != src25.width:

        raise ValueError(
            "2016 and 2025 raster widths are different."
        )

    if src16.height != src25.height:

        raise ValueError(
            "2016 and 2025 raster heights are different."
        )

    if src16.crs != src25.crs:

        raise ValueError(
            "2016 and 2025 CRS are different."
        )

    if not np.allclose(
        src16.transform,
        src25.transform
    ):

        raise ValueError(
            "2016 and 2025 spatial transforms are different."
        )

    print(
        "[OK] Dimensions:",
        src16.width,
        "×",
        src16.height
    )

    print(
        "[OK] CRS:",
        src16.crs
    )

    print(
        "[OK] Spatial grid matches."
    )

    print(
        "[OK] Pixel size:",
        src16.transform.a,
        "×",
        abs(src16.transform.e)
    )


# =============================================================================
# 10. CREATE RECLASSIFIED RASTERS
# =============================================================================

print()
print("=" * 80)
print("CREATING RECLASSIFIED LULC RASTERS")
print("=" * 80)


with rasterio.open(LULC_2016) as src:

    profile = src.profile.copy()

    profile.update(
        dtype="uint8",
        count=1,
        nodata=255,
        compress="deflate"
    )

    with rasterio.open(
        RECLASS_2016_RASTER,
        "w",
        **profile
    ) as dst:

        for row_start in range(
            0,
            src.height,
            CHUNK_ROWS
        ):

            rows = min(
                CHUNK_ROWS,
                src.height - row_start
            )

            window = Window(
                0,
                row_start,
                src.width,
                rows
            )

            data = src.read(
                1,
                window=window,
                masked=True
            )

            arr = np.asarray(
                data.filled(255)
            )

            reclass = reclassify_array(
                arr
            )

            dst.write(
                reclass,
                1,
                window=window
            )


print(
    "[CREATED]",
    os.path.basename(
        RECLASS_2016_RASTER
    )
)


with rasterio.open(LULC_2025) as src:

    profile = src.profile.copy()

    profile.update(
        dtype="uint8",
        count=1,
        nodata=255,
        compress="deflate"
    )

    with rasterio.open(
        RECLASS_2025_RASTER,
        "w",
        **profile
    ) as dst:

        for row_start in range(
            0,
            src.height,
            CHUNK_ROWS
        ):

            rows = min(
                CHUNK_ROWS,
                src.height - row_start
            )

            window = Window(
                0,
                row_start,
                src.width,
                rows
            )

            data = src.read(
                1,
                window=window,
                masked=True
            )

            arr = np.asarray(
                data.filled(255)
            )

            reclass = reclassify_array(
                arr
            )

            dst.write(
                reclass,
                1,
                window=window
            )


print(
    "[CREATED]",
    os.path.basename(
        RECLASS_2025_RASTER
    )
)


# =============================================================================
# 11. INITIALIZE TRANSITION STATISTICS
# =============================================================================

transition_pixels = {}

transition_area_km2 = {}

for from_class in CLASS_NAMES:

    for to_class in CLASS_NAMES:

        key = (
            from_class,
            to_class
        )

        transition_pixels[key] = 0

        transition_area_km2[key] = 0.0


# =============================================================================
# 12. PROCESS PIXELS
# =============================================================================

print()
print("=" * 80)
print("CALCULATING 2016 → 2025 TRANSITIONS")
print("=" * 80)


with rasterio.open(LULC_2016) as src16, \
     rasterio.open(LULC_2025) as src25:

    profile = src16.profile.copy()

    profile.update(
        dtype="uint8",
        count=1,
        nodata=255,
        compress="deflate"
    )

    with rasterio.open(
        TRANSITION_RASTER,
        "w",
        **profile
    ) as transition_dst:

        for row_start in range(
            0,
            src16.height,
            CHUNK_ROWS
        ):

            rows = min(
                CHUNK_ROWS,
                src16.height - row_start
            )

            window = Window(
                0,
                row_start,
                src16.width,
                rows
            )

            data16 = src16.read(
                1,
                window=window,
                masked=True
            )

            data25 = src25.read(
                1,
                window=window,
                masked=True
            )

            mask16 = np.ma.getmaskarray(
                data16
            )

            mask25 = np.ma.getmaskarray(
                data25
            )

            valid = (
                (~mask16)
                &
                (~mask25)
            )

            arr16 = np.asarray(
                data16.filled(255)
            )

            arr25 = np.asarray(
                data25.filled(255)
            )

            lulc16 = reclassify_array(
                arr16
            )

            lulc25 = reclassify_array(
                arr25
            )

            valid &= (
                (lulc16 != 255)
                &
                (lulc25 != 255)
            )

            # -----------------------------------------------------------------
            # Transition raster
            #
            # Transition code:
            #
            # FROM × 10 + TO
            #
            # Example:
            #
            # Vegetation → Built
            #
            # 1 × 10 + 5 = 15
            # -----------------------------------------------------------------

            transition_out = np.full(
                lulc16.shape,
                255,
                dtype=np.uint8
            )

            transition_out[valid] = (
                lulc16[valid] * 10
                +
                lulc25[valid]
            )

            transition_dst.write(
                transition_out,
                1,
                window=window
            )

            # -----------------------------------------------------------------
            # Calculate areas
            # -----------------------------------------------------------------

            for local_row in range(rows):

                global_row = (
                    row_start
                    +
                    local_row
                )

                pixel_area = (
                    calculate_row_pixel_area_km2(
                        src16,
                        global_row
                    )
                )

                valid_row = valid[
                    local_row
                ]

                row16 = lulc16[
                    local_row
                ]

                row25 = lulc25[
                    local_row
                ]

                for from_class in CLASS_NAMES:

                    for to_class in CLASS_NAMES:

                        transition_mask = (
                            valid_row
                            &
                            (row16 == from_class)
                            &
                            (row25 == to_class)
                        )

                        count = int(
                            np.count_nonzero(
                                transition_mask
                            )
                        )

                        if count > 0:

                            key = (
                                from_class,
                                to_class
                            )

                            transition_pixels[key] += count

                            transition_area_km2[key] += (
                                count
                                *
                                pixel_area
                            )


print()
print("[OK] Transition calculation completed.")


# =============================================================================
# 13. CREATE TRANSITION STATISTICS TABLE
# =============================================================================

print()
print("=" * 80)
print("CREATING TRANSITION STATISTICS TABLE")
print("=" * 80)


rows = []

for from_class in CLASS_NAMES:

    source_total_km2 = sum(

        transition_area_km2[
            (
                from_class,
                to_class
            )
        ]

        for to_class in CLASS_NAMES
    )

    for to_class in CLASS_NAMES:

        key = (
            from_class,
            to_class
        )

        area_km2 = (
            transition_area_km2[key]
        )

        area_ha = (
            area_km2 * 100.0
        )

        pixels = (
            transition_pixels[key]
        )

        if source_total_km2 > 0:

            source_percent = (
                area_km2
                /
                source_total_km2
                *
                100.0
            )

        else:

            source_percent = np.nan

        if from_class == to_class:

            transition_type = (
                "Persistence"
            )

        else:

            transition_type = (
                "Conversion"
            )

        rows.append({

            "From_Code":
                from_class,

            "From_Class":
                CLASS_NAMES[from_class],

            "To_Code":
                to_class,

            "To_Class":
                CLASS_NAMES[to_class],

            "Transition":
                (
                    CLASS_NAMES[from_class]
                    +
                    " → "
                    +
                    CLASS_NAMES[to_class]
                ),

            "Transition_Type":
                transition_type,

            "Pixels":
                pixels,

            "Area_ha":
                area_ha,

            "Area_km2":
                area_km2,

            "Percent_of_2016_Source":
                source_percent,
        })


transition_df = pd.DataFrame(
    rows
)


# =============================================================================
# 14. RANK TRANSITIONS
# =============================================================================

transition_df = transition_df.sort_values(
    by="Area_ha",
    ascending=False
).reset_index(
    drop=True
)

transition_df.insert(
    0,
    "Rank",
    np.arange(
        1,
        len(transition_df) + 1
    )
)


# =============================================================================
# 15. SAVE FULL TRANSITION TABLE
# =============================================================================

transition_df.to_csv(
    TRANSITION_CSV,
    index=False
)

print(
    "[CREATED]",
    TRANSITION_CSV
)


# =============================================================================
# 16. CONVERSION-ONLY TABLE
# =============================================================================

conversion_df = (
    transition_df[
        transition_df[
            "From_Code"
        ]
        !=
        transition_df[
            "To_Code"
        ]
    ]
    .copy()
)

conversion_df = (
    conversion_df
    .sort_values(
        "Area_ha",
        ascending=False
    )
    .reset_index(drop=True)
)

conversion_df.insert(
    0,
    "Conversion_Rank",
    np.arange(
        1,
        len(conversion_df) + 1
    )
)

conversion_csv = os.path.join(
    OUTPUT_DIR,
    "TZPR_NLP_LULC_Major_Conversion_Pathways_2016_2025.csv"
)

conversion_df.to_csv(
    conversion_csv,
    index=False
)

print(
    "[CREATED]",
    conversion_csv
)


# =============================================================================
# 17. TRANSITION MATRIX — HECTARES
# =============================================================================

matrix_ha = np.zeros(
    (
        len(CLASS_NAMES),
        len(CLASS_NAMES)
    )
)

for i in CLASS_NAMES:

    for j in CLASS_NAMES:

        matrix_ha[i, j] = (
            transition_area_km2[
                (i, j)
            ]
            *
            100.0
        )


matrix_ha_df = pd.DataFrame(
    matrix_ha,
    index=[
        CLASS_NAMES[i]
        for i in CLASS_NAMES
    ],
    columns=[
        CLASS_NAMES[i]
        for i in CLASS_NAMES
    ]
)

matrix_ha_df.index.name = (
    "2016_Class"
)

matrix_ha_df.to_csv(
    TRANSITION_MATRIX_CSV
)

print(
    "[CREATED]",
    TRANSITION_MATRIX_CSV
)


# =============================================================================
# 18. TRANSITION MATRIX — PERCENTAGE
# =============================================================================
#
# Percentage is calculated relative to the total 2016 area of each source
# class.
#
# Therefore each row should approximately sum to 100%.
#
# =============================================================================

matrix_percent = np.zeros(
    (
        len(CLASS_NAMES),
        len(CLASS_NAMES)
    )
)

for i in CLASS_NAMES:

    row_total = sum(

        transition_area_km2[
            (i, j)
        ]

        for j in CLASS_NAMES

    )

    for j in CLASS_NAMES:

        if row_total > 0:

            matrix_percent[i, j] = (
                transition_area_km2[
                    (i, j)
                ]
                /
                row_total
                *
                100.0
            )

        else:

            matrix_percent[i, j] = np.nan


matrix_percent_df = pd.DataFrame(
    matrix_percent,
    index=[
        CLASS_NAMES[i]
        for i in CLASS_NAMES
    ],
    columns=[
        CLASS_NAMES[i]
        for i in CLASS_NAMES
    ]
)

matrix_percent_df.index.name = (
    "2016_Class"
)

matrix_percent_df.to_csv(
    TRANSITION_PERCENT_CSV
)

print(
    "[CREATED]",
    TRANSITION_PERCENT_CSV
)


# =============================================================================
# 19. REQUESTED MAJOR TRANSITION RASTERS
# =============================================================================
#
# Requested:
#
# Trees → Built
# Crops → Built
# Trees → Crops
# Crops → Bare
# Grass → Built
# Shrub → Built
#
# Because Trees + Flooded Vegetation were combined:
#
# Trees → Built
# becomes
# Vegetation → Built
#
# Trees → Crops
# becomes
# Vegetation → Crops
#
# Crops → Bare
# becomes
# Crops → Bare/Snow-Ice
#
# =============================================================================


REQUESTED_TRANSITIONS = {

    "Vegetation_to_Built":
        (1, 5),

    "Crops_to_Built":
        (3, 5),

    "Vegetation_to_Crops":
        (1, 3),

    "Crops_to_Bare":
        (3, 6),

    "Grass_to_Built":
        (2, 5),

    "Shrub_to_Built":
        (4, 5),
}


# =============================================================================
# 20. CREATE MAJOR TRANSITION RASTERS
# =============================================================================

print()
print("=" * 80)
print("CREATING MAJOR TRANSITION RASTERS")
print("=" * 80)


with rasterio.open(
    LULC_2016
) as src16, \
     rasterio.open(
         LULC_2025
     ) as src25:

    profile = src16.profile.copy()

    profile.update(
        dtype="uint8",
        count=1,
        nodata=0,
        compress="deflate"
    )

    for transition_name, (
        from_class,
        to_class
    ) in REQUESTED_TRANSITIONS.items():

        output_path = os.path.join(
            OUTPUT_DIR,
            f"TZPR_NLP_{transition_name}_2016_2025.tif"
        )

        print()
        print(
            "Creating:",
            transition_name
        )

        with rasterio.open(
            output_path,
            "w",
            **profile
        ) as dst:

            for row_start in range(
                0,
                src16.height,
                CHUNK_ROWS
            ):

                rows_count = min(
                    CHUNK_ROWS,
                    src16.height - row_start
                )

                window = Window(
                    0,
                    row_start,
                    src16.width,
                    rows_count
                )

                data16 = src16.read(
                    1,
                    window=window,
                    masked=True
                )

                data25 = src25.read(
                    1,
                    window=window,
                    masked=True
                )

                mask16 = np.ma.getmaskarray(
                    data16
                )

                mask25 = np.ma.getmaskarray(
                    data25
                )

                valid = (
                    (~mask16)
                    &
                    (~mask25)
                )

                arr16 = np.asarray(
                    data16.filled(255)
                )

                arr25 = np.asarray(
                    data25.filled(255)
                )

                re16 = reclassify_array(
                    arr16
                )

                re25 = reclassify_array(
                    arr25
                )

                transition = (
                    (re16 == from_class)
                    &
                    (re25 == to_class)
                    &
                    valid
                )

                output = np.zeros(
                    re16.shape,
                    dtype=np.uint8
                )

                output[
                    transition
                ] = 1

                dst.write(
                    output,
                    1,
                    window=window
                )

        print(
            "[CREATED]",
            os.path.basename(
                output_path
            )
        )


# =============================================================================
# 21. REQUESTED TRANSITION SUMMARY
# =============================================================================

requested_rows = []

for transition_name, (
    from_class,
    to_class
) in REQUESTED_TRANSITIONS.items():

    row = transition_df[
        (
            transition_df[
                "From_Code"
            ]
            ==
            from_class
        )
        &
        (
            transition_df[
                "To_Code"
            ]
            ==
            to_class
        )
    ]

    if len(row) > 0:

        r = row.iloc[0]

        requested_rows.append({

            "Transition":
                r["Transition"],

            "Area_ha":
                r["Area_ha"],

            "Area_km2":
                r["Area_km2"],

            "Percent_of_2016_Source":
                r["Percent_of_2016_Source"],

            "Raster":
                (
                    f"TZPR_NLP_"
                    f"{transition_name}"
                    f"_2016_2025.tif"
                ),

        })


requested_df = pd.DataFrame(
    requested_rows
)

requested_df = (
    requested_df
    .sort_values(
        "Area_ha",
        ascending=False
    )
    .reset_index(drop=True)
)

requested_csv = os.path.join(
    OUTPUT_DIR,
    "TZPR_NLP_Requested_Major_Transition_Pathways.csv"
)

requested_df.to_csv(
    requested_csv,
    index=False
)

print()
print(
    "[CREATED]",
    requested_csv
)


# =============================================================================
# 22. PRINT TOP TRANSITIONS
# =============================================================================

print()
print("=" * 80)
print("TOP LULC CONVERSION PATHWAYS")
print("=" * 80)

print()

for _, row in conversion_df.head(15).iterrows():

    print(
        f"{int(row['Conversion_Rank']):02d}. "
        f"{row['Transition']:<35} "
        f"{row['Area_ha']:>12,.2f} ha "
        f"({row['Area_km2']:>10,.2f} km²)"
    )


# =============================================================================
# 23. MANUSCRIPT SUMMARY
# =============================================================================

with open(
    SUMMARY_TXT,
    "w",
    encoding="utf-8"
) as f:

    f.write("=" * 80 + "\n")

    f.write(
        "SECTION 4.6 — MAJOR LULC TRANSITION PATHWAYS\n"
    )

    f.write("=" * 80 + "\n\n")

    f.write(
        "Comparison period: 2016 → 2025\n\n"
    )

    f.write(
        "ANALYSIS CLASSES\n"
    )

    f.write(
        "-" * 80 + "\n"
    )

    for code, name in CLASS_NAMES.items():

        f.write(
            f"{code} = {name}\n"
        )

    f.write("\n")

    f.write(
        "TOP 15 CONVERSION PATHWAYS\n"
    )

    f.write(
        "-" * 80 + "\n"
    )

    for _, row in conversion_df.head(15).iterrows():

        f.write(
            f"{int(row['Conversion_Rank']):02d}. "
            f"{row['Transition']} : "
            f"{row['Area_ha']:,.2f} ha "
            f"({row['Area_km2']:,.2f} km²) "
            f"| "
            f"{row['Percent_of_2016_Source']:.2f}% "
            f"of source class\n"
        )

    f.write("\n")

    f.write(
        "REQUESTED MAJOR TRANSITIONS\n"
    )

    f.write(
        "-" * 80 + "\n"
    )

    for _, row in requested_df.iterrows():

        f.write(
            f"{row['Transition']} : "
            f"{row['Area_ha']:,.2f} ha "
            f"({row['Area_km2']:,.2f} km²) "
            f"| "
            f"{row['Percent_of_2016_Source']:.2f}% "
            f"of source class\n"
        )

    f.write("\n")

    f.write(
        "INTERPRETATION\n"
    )

    f.write(
        "-" * 80 + "\n"
    )

    f.write(
        "Transition areas represent pixels whose LULC class "
        "changed from the source class in 2016 to the destination "
        "class in 2025. Persistence transitions (same class in "
        "2016 and 2025) are excluded when ranking conversion "
        "pathways.\n\n"
    )

    f.write(
        "Trees and flooded vegetation were grouped into a single "
        "Vegetation category for the transition analysis. Bare "
        "land and snow/ice were grouped into a single Bare/Snow-Ice "
        "category.\n\n"
    )

    f.write(
        "The transition analysis identifies spatial associations "
        "between LULC classes and does not by itself establish "
        "causality. In particular, conversions involving built-up "
        "land should not automatically be attributed to highway "
        "development without supporting spatial or distance-gradient "
        "analysis.\n"
    )


# =============================================================================
# 24. FINAL OUTPUT LIST
# =============================================================================

print()
print("=" * 80)
print("ANALYSIS COMPLETED")
print("=" * 80)

print()
print("Output directory:")
print(OUTPUT_DIR)

print()
print("Generated files:")

for filename in sorted(
    os.listdir(OUTPUT_DIR)
):

    path = os.path.join(
        OUTPUT_DIR,
        filename
    )

    if os.path.isfile(path):

        size_mb = (
            os.path.getsize(path)
            /
            (1024 * 1024)
        )

        print(
            f"[OK] {filename:<70} "
            f"{size_mb:>8.2f} MB"
        )


print()
print("=" * 80)
print("SECTION 4.6 DATA EXTRACTION FINISHED")
print("=" * 80)

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive
